In [ ]:
from google.colab import drive
from osgeo import ogr
import os

# Подключаем Google Drive
drive.mount("/content/drive", force_remount=True)

# 1. Открываем файл stations.geojson
data_source = ogr.Open('/content/drive/MyDrive/stations.geojson', 1)  # Режим редактирования
layer = data_source.GetLayerByIndex(0)

# 2. Проверяем наличие поля "subway"
layer_defn = layer.GetLayerDefn()
field_names = [layer_defn.GetFieldDefn(i).GetName() for i in range(layer_defn.GetFieldCount())]
print("Доступные поля:", field_names)

if 'subway' not in field_names:
    raise ValueError("Отсутствует поле 'subway'")

# 3. Создаем поле "x_coordinate" (если еще не существует)
if 'x_coordinate' not in field_names:
    field_defn = ogr.FieldDefn("x_coordinate", ogr.OFTReal)
    layer.CreateField(field_defn)

# 4. Фильтруем станции: subway != 'yes' ИЛИ subway IS NULL ИЛИ subway = ''
sql_filter = "subway != 'yes' OR subway IS NULL OR subway = ''"
layer.SetAttributeFilter(sql_filter)
print(f"Применен фильтр: {sql_filter}")

# 5. Создаем новый GeoJSON-файл для результатов
output_path = "/content/drive/MyDrive/2_Семёнова_.json"
if os.path.exists(output_path):
    os.remove(output_path)  # Удаляем старый файл, если существует

driver = ogr.GetDriverByName("GeoJSON")
output_ds = driver.CreateDataSource(output_path)
output_layer = output_ds.CreateLayer("non_subway_stations", geom_type=ogr.wkbPoint)

# Копируем все поля из исходного слоя
for i in range(layer_defn.GetFieldCount()):
    output_layer.CreateField(layer_defn.GetFieldDefn(i))

# 6. Обрабатываем отфильтрованные станции
filtered_count = 0
layer.ResetReading()

for feat in layer:
    geom = feat.GetGeometryRef()
    if geom is not None:
        # Создаем новый объект для выходного слоя
        new_feat = ogr.Feature(output_layer.GetLayerDefn())
        new_feat.SetGeometry(geom)

        # Копируем все атрибуты
        for i in range(feat.GetFieldCount()):
            new_feat.SetField(i, feat.GetField(i))

        # Записываем координату X
        new_feat.SetField("x_coordinate", geom.GetX())
        output_layer.CreateFeature(new_feat)
        filtered_count += 1

print(f"Обработано станций: {filtered_count}")

# 7. Закрываем ресурсы
output_ds = None
data_source = None

print(f"Результат сохранен в: {output_path}")


Mounted at /content/drive
Доступные поля: ['full_id', 'osm_id', 'subway', 'station', 'depth', 'colour', 'network', 'train', 'public_transport', 'operator', 'name', 'x_coordinate']
Применен фильтр: subway != 'yes' OR subway IS NULL OR subway = ''
Обработано станций: 73
Результат сохранен в: /content/drive/MyDrive/2_Семёнова_.json
